# Proximal Policy Optimization(PPO)

## Why PPO?

- PPO was developed inorder to solve the **Stability V/S Sample Efficieny** trade-off.

- PPO ensures that the new policy stays **proximal(close)** to the old policy. This prevents the "policy collapse".

- PPO uses a **Clipped-Surrogate Objective**

## How PPO?

- **The Ratio(rt)**:- We track how much more(or less) likely an action is under the new policy compared to the old one.

- **The Clip**:- If the ratio starts drifting too far away from 1(e.g. beyond 0.8 or 1.2), we *clip* it.

- **The Result**:- If an action was great, we want to increase its probability, but only up to a certain point. If the update is too large, the "clip" removes the incentive to push the change further. This results in flattening of the gradients when the change is too drastic.

## Code PPO

#### Import the Libraries & Set-up Config

In [1]:
import  torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
import numpy as np 
import gymnasium as gym 
import wandb
from gymnasium.wrappers import RecordVideo
import os 

In [16]:
class Config:
    def __init__(self):
        self.env_name = "Pendulum-v1"
        self.total_timesteps = 50000
        self.learning_rate = 3e-4
        self.gamma = 0.99
        self.gae_lambda = 0.95
        self.clip_epsilon = 0.2
        self.epochs = 10
        self.batch_size = 64
        self.buffer_size = 2048
        self.hidden_size = 64
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
config = Config()
wandb.init(project = "simple_ppo_from_scratch", config = vars(config), mode = "online")

#### Actor-Critic Network of the PPO

- **The Actor**: Decides which action to take(the Policy).
- **The Critic**: Predicts how much reward the agent will get from a certain state(Value Function)

In [31]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, action_dim):
        ''' 
            We make a Linear Neural Network
            Because the Action-Space of our Environment is vectors(numbers)
        '''
        super(ActorCritic, self).__init__()
        
        #We make a Linear Neural Network
        self.shared = nn.Sequential(
            nn.Linear(obs_dim, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh()
        )
        self.actor_mean = nn.Linear(config.hidden_size, action_dim)
        self.actor_logstd = nn.Parameter(torch.zeros(action_dim))
        self.critic = nn.Linear(config.hidden_size, 1)
        
    def forward(self, x):
        
        #If x is 1-D, unsqueeze it to have a batch dim of [1, dim]
        if x.dim() == 1:
            x = x.unsqueeze(0)
        
        x = self.shared(x)
        mean = self.actor_mean(x)
        logstd = self.actor_logstd.expand_as(mean)
        std = torch.exp(logstd)
        value = self.critic(x)
        return mean, std, value
    
    def get_action(self, obs):
        #Ensure obs is a 1-D Array
        obs = np.array(obs).flatten()
        
        obs_tensor = torch.FloatTensor(obs).to(config.device)
        mean, std, value = self.forward(obs_tensor)
        dist = torch.distributions.Normal(mean, std)
        action = dist.sample()
        log_prob = dist.log_prob(action).sum(dim = -1)
        
        #Squeeze the Action to remove batch-Dim before giving it back to the ENV
        action = action.squeeze(0)
        
        return action.cpu().detach().numpy(), log_prob.cpu().detach().item(), value.cpu().detach().item()
    
    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = torch.distributions.Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim = -1)
        entropy = dist.entropy().sum(dim = -1)
        return log_prob, value.squeeze(-1), entropy
        

### Rollout Buffer

In [32]:
class RolloutBuffer:
    def __init__(self):
        self.obs = []
        self.actions = []
        self.rewards = []
        self.dones = []
        self.log_probs = []
        self.values = []
        
    def add(self, obs, action, reward, done, log_prob, value):
        self.obs.append(obs)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)
        
    def get(self):
        data = {
            "obs":torch.FloatTensor(np.array(self.obs)).to(config.device),
            "actions":torch.FloatTensor(np.array(self.actions)).to(config.device),
            "rewards":torch.FloatTensor(np.array(self.rewards)).to(config.device),
            "dones":torch.FloatTensor(np.array(self.dones)).to(config.device),
            "log_probs":torch.FloatTensor(np.array(self.log_probs)).to(config.device),
            "values":torch.FloatTensor(np.array(self.values)).to(config.device)
        }
        self.clear()
        return data

    def clear(self):
        self.obs, self.actions, self.rewards = [], [], []
        self.dones, self.log_probs, self.values = [], [], []

In [33]:
def compute_gae(buffer_data, last_value):
    rewards = buffer_data["rewards"]
    values = buffer_data["values"]
    dones = buffer_data["dones"]
    advantages = torch.zeros_like(rewards).to(config.device)
    last_gae = 0
    
    for t in reversed(range(len(rewards))):
        if t == len(rewards) - 1:
            next_value = last_value
        else:
            next_value = values[t + 1]
        delta = rewards[t] + config.gamma * next_value * (1 - dones[t]) - values[t]
        last_gae = delta + config.gamma * config.gae_lambda * (1 - dones[t]) * last_gae
        advantages[t] = last_gae
        
    returns = advantages + values
    return advantages, returns 

def ppo_update(policy, optimizer, buffer_data, advantages, returns):
    obs = buffer_data['obs']
    actions = buffer_data['actions']
    old_log_probs = buffer_data['log_probs']
    
    #Normalize Advantage (Crucial for stability)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    dataset_size = len(obs)
    indices = np.arange(dataset_size)
    
    policy_losses, value_losses, entropies = [], [], []
    
    for _ in range(config.epochs):
        np.random.shuffle(indices)
        for start in range(0, dataset_size, config.batch_size):
            end = start + config.batch_size
            batch_idx = indices[start:end]
            
            b_obs = obs[batch_idx]
            b_actions = actions[batch_idx]
            b_old_log_probs = old_log_probs[batch_idx]
            b_advantages = advantages[batch_idx]
            b_returns = returns[batch_idx]
            
            log_prob, value, entropy = policy.evaluate(b_obs, b_actions)
            
            ratio = torch.exp(log_prob - b_old_log_probs)
            surr1 = ratio * b_advantages
            surr2 = torch.clamp(ratio, 1 - config.clip_epsilon, 1 + config.clip_epsilon) * b_advantages
            policy_loss = -torch.min(surr1, surr2).mean()
            
            value_loss = nn.MSELoss()(value,b_returns)
            entropy_loss = entropy.mean()
            
            #Total Loss => Minimize Policy Loss, Minimize Value Loss, Maximize Entropy(-entropy_loss)
            loss = policy_loss + 0.5 * value_loss - 0.01 * entropy_loss
            
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
            optimizer.step()
            
            policy_losses.append(policy_loss.item())
            value_losses.append(value_loss.item())
            entropies.append(entropy_loss.item())
            
    return np.mean(policy_losses), np.mean(value_losses), np.mean(entropies) 

### Training Loop

In [34]:
def train():
    video_dir = "./lunar_ppo_videos"
    os.makedirs(video_dir, exist_ok=True)
    env = gym.make(config.env_name, render_mode = "rgb_array")
    
    #Function to record video at every interval of 50 episodes
    def video_trigger(episode_id):
        return episode_id % 50 == 0
    
    env = RecordVideo(env, video_folder = video_dir, episode_trigger=video_trigger, name_prefix="ppo_agent")
    
    obs_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    
    policy = ActorCritic(obs_dim, action_dim).to(config.device)
    optimizer = optim.Adam(policy.parameters(), lr = config.learning_rate)
    buffer = RolloutBuffer()
    
    obs, _ = env.reset()
    episode_reward = 0
    episode_length = 0
    episode_count = 0
    
    print(f"Starting training for {config.total_timesteps} timesteps..")
    
    for timestep in range(1, config.total_timesteps + 1):
        #===========Step-1 COLLECT DATA==============
        action, log_prob, value = policy.get_action(obs)
        
        #-----------Step the ENV---------------
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        #------------Store in BUFFER--------------
        buffer.add(obs, action, reward, done, log_prob, value)
        
        obs = next_obs
        episode_reward += reward 
        episode_length += 1
        
        #=============Step-2 UPDATE PPO============
        if timestep % config.buffer_size == 0:
            #Calculate value of last state for GAE
            with torch.no_grad():
                _, _, last_value = policy(torch.FloatTensor(obs).to(config.device))
                last_value = last_value.cpu().item()
                
            buffer_data = buffer.get()
            advantages, returns = compute_gae(buffer_data, last_value)
            
            #-----------Update the POLICY-----------------
            avg_pol_loss, avg_val_loss, avg_entropy = ppo_update(policy, optimizer, buffer_data, advantages, returns)
            
            #-------------LOGG IT---------------------
            wandb.log({
                "timesteps": timestep,
                "update/policy_loss": avg_pol_loss,
                "update/value_loss": avg_val_loss,
                "update/entropy": avg_entropy,
                "update/learning_rate": config.learning_rate
            }) 
        
        #================Step-3 EPISODE End Logging=============
        if done:
            episode_count += 1
            wandb.log({
                "episode": episode_count,
                "episode_reward": episode_reward,
                "episode_length": episode_length
            })
            
            #Check video generated?
            expected_video_path = os.path.join(video_dir, f"ppo_agent-episode-{episode_count - 1}.mp4")
            
            if os.path.exists(expected_video_path):
                print(f"Found video for episode: {episode_count - 1}, logging to WandB....")
                wandb.log({
                    "gameplay": wandb.Video(expected_video_path, fps = 30, format = "mp4")
                })
            
            if episode_count % 10 == 0:
                print(f"Timestep: {timestep} | Episode: {episode_count} | Reward: {episode_reward:.2f}")
            
            obs, _ = env.reset()
            episode_reward = 0
            episode_length = 0
            
    env.close()
    wandb.finish()
    print("Training Complete!")

In [35]:
train()

c:\Python\Lib\site-packages\gymnasium\wrappers\rendering.py:283: UserWarning: WARN: Overwriting existing videos at d:\Atharva\Implementations\From_Scratch\Reinforcement_Learning\PPO\lunar_ppo_videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Starting training for 50000 timesteps..
Timestep: 2000 | Episode: 10 | Reward: -1753.97
Timestep: 4000 | Episode: 20 | Reward: -1688.00
Timestep: 6000 | Episode: 30 | Reward: -1584.95
Timestep: 8000 | Episode: 40 | Reward: -1460.38
Timestep: 10000 | Episode: 50 | Reward: -1626.38
Timestep: 12000 | Episode: 60 | Reward: -1078.28
Timestep: 14000 | Episode: 70 | Reward: -1190.58
Timestep: 16000 | Episode: 80 | Reward: -1207.68
Timestep: 18000 | Episode: 90 | Reward: -1365.37
Timestep: 20000 | Episode: 100 | Reward: -966.52
Timestep: 22000 | Episode: 110 | Reward: -1743.07
Timestep: 24000 | Episode: 120 | Reward: -1650.00
Timestep: 26000 | Episode: 130 | Reward: -1081.60
Timestep: 28000 | Episode: 140 | Reward: -1344.47
Timestep: 30000 | Episode: 150 | Reward: -1065.10
Timestep: 32000 | Episode: 160 | Reward: -968.37
Timestep: 34000 | Episode: 170 | Reward: -959.02
Timestep: 36000 | Episode: 180 | Reward: -903.25
Timestep: 38000 | Episode: 190 | Reward: -993.55
Timestep: 40000 | Episode: 2

episode,▁▁▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
episode_length,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
episode_reward,▂▇▅▇▂▅▄▃█▄▅▅▃▆▆▃▆▆▆▆▂▄▁▃▇▆▆▆▄▇▇▅▃▇▆▆▆▇▄▁
timesteps,▁▁▂▂▂▃▃▃▃▄▄▄▅▅▅▆▆▆▆▇▇▇██
update/entropy,▇▇▇▆▆▇██▇▇▇▆▇▆▅▄▄▅▅▅▅▄▃▁
update/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
update/policy_loss,▅▆▃██▇▆▅▁▇▅▅█▄▂▅▄█▄▇▇▂▃▄
update/value_loss,█▄▆▄▃▃▅▃▂▃▃▃▁▂▂▂▂▁▂▂▂▁▂▂
episode,250
episode_length,200
episode_reward,-987.79474


Training Complete!
